In [ ]:
import torch
from dataloader.mnist import load_test_data

In [2]:
model = torch.load('./results/bnn_mnist_1_hidden_0_batch_acc_90_37_sgd_lr_001.pth.tar', weights_only=False)

In [ ]:
model['state_dict'].keys()

odict_keys(['classifier.1.weight', 'classifier.1.weight_org', 'classifier.3.weight', 'classifier.3.weight_org'])

In [4]:
model['state_dict']['classifier.1.weight'].size()

torch.Size([256, 784])

In [5]:
def quantize(x):
  '''get binary sign'''
  return (x>=0).to(x.dtype)*2-1

def binarize(x):
  '''turn +/-1 to 1/0 bit'''
  return x.clone().apply_(lambda val: float(1) if val == 1 else float(0)).to(torch.uint8)

def debinarize(x):
  '''turn 1/0 to +/-1 bit'''
  return x*2-1

In [6]:
w_1 = quantize(model['state_dict']['classifier.1.weight'])
w_2 = quantize(model['state_dict']['classifier.3.weight'])

In [ ]:
w_1

tensor([[ 1.,  1., -1.,  ...,  1.,  1.,  1.],
        [-1., -1., -1.,  ..., -1., -1., -1.],
        [-1.,  1., -1.,  ..., -1.,  1.,  1.],
        ...,
        [-1., -1.,  1.,  ..., -1.,  1., -1.],
        [-1., -1., -1.,  ..., -1.,  1.,  1.],
        [-1.,  1.,  1.,  ...,  1., -1., -1.]])

In [8]:
w_1q = binarize(w_1)
w_2q = binarize(w_2)

In [9]:
def feedforward(input):
  input = input.flatten()
  input = debinarize(input)
  
  layer_1 = w_1 @ input
  layer_1q = quantize(layer_1)
  
  out = w_2 @ layer_1q

  return torch.argmax(out)

In [10]:
import time

def test(forward_pass):
  test_loader = load_test_data()
  data_iter = iter(test_loader)
  incorrect = 0
  size = 0
  i = 0
  total_time = 0
  iterations = len(data_iter)
  for data, label in data_iter:
    size += data.size()[0]
    for x, y in zip(data, label):
      start = time.time_ns()
      out = forward_pass(x)
      end = time.time_ns()
      total_time += end - start
      if out.item() != y.item():
        incorrect += 1
    i += 1
    print(f"{i}/{iterations}")
      
  avg_time = total_time / size * 10**-6 
  frequency = 1/(avg_time*10**-3)
  print(f" Accuracy = {((size - incorrect) / size) * 100}%")
  print(f" Average Time Per Image = {avg_time} (ms)")
  print(f"Images per sec = {frequency} Hz")

In [11]:
test(feedforward)

/Users/josephattalla/projects/hardware-binary-neural-network/models/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


1/10
2/10
3/10
4/10
5/10
6/10
7/10
8/10
9/10
10/10
 Accuracy = 90.36999999999999%
 Average Time Per Image = 0.025653 (ms)
Images per sec = 38981.7955015008 Hz


In [12]:
def xnor(a, b):
  out = ((~(a ^ b)) & 1).item()
  if out != 1 and out != 0:
    print("error out = ", out)
  return out

In [34]:
# A = weights, size = [output_size, input_size]
# B = input
def xnor_matmul(w, x):
  # out_size, in_size = w.size()
  # out = torch.zeros((out_size,))

  # for i in range(out_size):
  #   popcount = 0
  #   for j in range(in_size):
  #     popcount += xnor(w[i][j], x[j])

  #   out[i] = popcount*2 - in_size
  out = (~(w^x) & 1).sum(axis=-1)*2-w.size()[1]

  return out

In [30]:
print(x.sum())

tensor(71.)


In [14]:
test_loader = load_test_data()
data_iter = iter(test_loader)
data, label = next(data_iter)
x = data[0].flatten()
y = label[0]

In [26]:
layer2 = w_1 @ (x*2-1)

In [27]:
layer2

tensor([ -58.,  -38., -100.,  -80.,  -28.,  -54.,  -54.,  -80.,  -58.,  -64.,
         -62.,  -80.,  -38.,   12.,  -52.,  -64.,  -54.,  -72.,  -44.,  -50.,
         -52.,  -62.,  -46.,  -50.,  -64.,  -82.,  -74.,  -54.,  -70.,  -64.,
         -54.,  -70.,  -66.,  -54.,  -82.,  -86.,  -66.,   48.,  -78.,  -54.,
         -86.,  -88.,  -74.,  -86.,  -36.,   12.,  -86.,  -60.,  -82.,  -60.,
         -56.,   72.,  -56.,  -26.,  -50.,  -70.,  -66.,  -46.,  -38.,  -32.,
         -60.,  -76.,  -48.,  -18.,  -68.,  -44.,  -38.,  -52.,  -60.,   72.,
         -48.,  -58.,  -74.,  -74.,  -80.,  -30.,  -60.,  -50.,  -24.,  -56.,
         -68.,  -42.,  -54.,  -50.,  -68.,  -62.,  -30.,  -36.,  -44., -112.,
         -66.,  -60.,  -76.,  -74.,  -78.,  -60.,   12.,  -48.,  -32.,  -60.,
         -66.,  -72.,  -58.,  -52.,  -64.,  -44.,  -58.,  -70.,  -60.,  -64.,
         -78.,  -82.,  -84.,  -42.,  -72.,  -78.,  -28.,  -62.,  -44.,  -82.,
        -100., -102.,  -72.,  -54.,  -50.,  -62.,  -50.,  -44., 

In [35]:
layer2q = xnor_matmul(w_1q, x.to(torch.uint8))

In [17]:
torch.equal(layer2, layer2q)

True

In [18]:
layer2 = quantize(layer2)
out = w_2 @ layer2

In [19]:
torch.argmax(out)

tensor(7)

In [20]:
layer2q = binarize(quantize(layer2q))
outq = xnor_matmul(w_2q, layer2q)

In [21]:
torch.equal(out, outq)

True

In [37]:
def feedforward_q(input):
  input = input.flatten().to(torch.uint8)

  layer_1 = xnor_matmul(w_1q, input)

  layer_1q = binarize(quantize(layer_1))

  out = xnor_matmul(w_2q, layer_1q)
  
  return torch.argmax(out)

In [38]:
feedforward_q(x)

tensor(7)

In [39]:
test(feedforward_q)

/Users/josephattalla/projects/hardware-binary-neural-network/models/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


1/10
2/10
3/10
4/10
5/10
6/10
7/10
8/10
9/10
10/10
 Accuracy = 90.36999999999999%
 Average Time Per Image = 0.2796053 (ms)
Images per sec = 3576.4701169827613 Hz
